# OpenEnv SME Negotiator — GRPO Training on Colab

**Theme #1 — Multi-Agent Interactions** (Razorpay Fix-My-Itch #82.8: SME vs large-buyer payment-term negotiation).

This notebook follows the **canonical Meta-PyTorch OpenEnv + TRL pattern** (`rollout_func` + `generate_rollout_completions`, multiple verifiable reward functions, GRPO via TRL, Hugging Face Router for inference) demonstrated in:

* [meta-pytorch/OpenEnv → tutorial/04-training.md](https://github.com/meta-pytorch/OpenEnv/blob/main/tutorial/04-training.md)
* [meta-pytorch/OpenEnv → tutorial/examples/wordle.py](https://github.com/meta-pytorch/OpenEnv/blob/main/tutorial/examples/wordle.py)
* [TRL `openenv_wordle_grpo` reference notebook](https://github.com/huggingface/trl/blob/main/examples/notebooks/openenv_wordle_grpo.ipynb)

What you get end-to-end:
1. Install OpenEnv + TRL + Unsloth-compatible deps.
2. Reset the in-process `SMELiquidityEnvironment` (Theme 1/2/3.1).
3. Score a deterministic heuristic baseline on all three tasks.
4. GRPO fine-tune Qwen2.5-1.5B-Instruct with **four verifiable reward functions** (solvency · NPV · compliance · format).
5. Plot the reward curve and compare baseline vs trained.
6. Optional: push checkpoint + model card to the Hugging Face Hub.

## 1. Install dependencies

We pin TRL 0.29 because that is the latest release that exposes both `rollout_func=` and `trl.experimental.openenv.generate_rollout_completions`.

In [ ]:
%pip install -q -U pip
%pip install -q "trl==0.29.0" "transformers>=4.45,<4.60" "peft>=0.19.1" "datasets" "accelerate" "matplotlib" "huggingface_hub>=0.20" "trackio" "openenv-core"

In [ ]:
import os
from pathlib import Path

REPO_DIR = Path("OpenEnv_SME_Negotiator-2.o")
if not REPO_DIR.exists():
    !git clone -q https://github.com/SkandaGanesha1/OpenEnv_SME_Negotiator-2.o.git
%cd $REPO_DIR
%pip install -q -e .[rl]

## 2. Configuration

All LLM inference is routed through the **Hugging Face Router** (`https://router.huggingface.co/v1`). Add `HF_TOKEN` in Colab Secrets (the key icon in the left sidebar) so the trained policy can call the router during evaluation.

In [ ]:
import os
from pathlib import Path

# Silence experimental-API warnings.
os.environ.setdefault("TRL_EXPERIMENTAL_SILENCE", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Hugging Face Router (canonical for this hackathon).
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["OPENAI_API_KEY"] = hf_token
        os.environ["OPENAI_BASE_URL"] = "https://router.huggingface.co/v1"
        os.environ["API_BASE_URL"]   = "https://router.huggingface.co/v1"
except Exception:
    pass

MODEL_NAME    = "Qwen/Qwen2.5-1.5B-Instruct"   # fits a free-tier T4
TASK_NAME     = "liquidity-correlation-hard"
DIFFICULTY    = "hard"
TOTAL_PERIODS = 2
MAX_TURNS     = 12          # max env steps per rollout episode
MAX_NEW_TOKENS = 192        # per LLM turn; the action JSON is short
DATASET_SIZE  = 32          # small for a Colab demo; raise to 1k+ for real training
NUM_GENERATIONS = 4         # GRPO group size (>= 2 required)
TRAIN_STEPS   = 25
OUTPUT_DIR    = Path("outputs/grpo_sme_liquidity_colab")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"MODEL_NAME    = {MODEL_NAME}")
print(f"TASK_NAME     = {TASK_NAME} (difficulty={DIFFICULTY}, total_periods={TOTAL_PERIODS})")
print(f"DATASET_SIZE  = {DATASET_SIZE}, NUM_GENERATIONS = {NUM_GENERATIONS}, TRAIN_STEPS = {TRAIN_STEPS}")
print(f"OUTPUT_DIR    = {OUTPUT_DIR}")

## 3. Smoke-test the environment (Gym-style API)

Confirm `reset()` / `step()` / `state()` round-trip cleanly through the in-process `SMELiquidityEnvironment`. This mirrors `tutorial/01-environments.md` from OpenEnv.

In [ ]:
from server.environment import SMELiquidityEnvironment

env_smoke = SMELiquidityEnvironment(total_periods=TOTAL_PERIODS)
obs = env_smoke.reset(seed=42, difficulty=DIFFICULTY, task_name=TASK_NAME)
sme = env_smoke.state.world_state.smes[0]
print({
    "task_name": obs.task_name,
    "buyer_price": obs.buyer_price,
    "buyer_days": obs.buyer_days,
    "liquidity_threshold": obs.liquidity_threshold,
    "open_deals": obs.open_deal_ids,
    "initial_cash": sme.cash_balance,
    "required_minimum_cash": sme.required_minimum_cash,
    "current_period": f"{obs.current_period}/{obs.total_periods}",
})

## 4. Reproducible heuristic baseline on all three tasks

These deterministic baseline numbers are the “before” side of the judging criterion *Showing Improvement in Rewards (20%)*.

In [ ]:
from rl.demo import run_heuristic_episode

BASELINE_TASKS = [
    ("payment-terms-easy",        "easy"),
    ("payment-terms-medium",      "medium"),
    ("liquidity-correlation-hard", "hard"),
]
BASELINE_SEEDS = [11, 22, 33]

baseline = {}
for task, diff in BASELINE_TASKS:
    runs = [run_heuristic_episode(seed=s, total_periods=TOTAL_PERIODS, task_name=task, difficulty=diff) for s in BASELINE_SEEDS]
    avg = sum(r["summary"]["verifiable_reward"] for r in runs) / len(runs)
    baseline[task] = round(avg, 4)
    print(f"  baseline {task:30s} avg_verifiable_reward={avg:.4f}")

## 5. Canonical OpenEnv + TRL `rollout_func`

This is the heart of the training loop. It is the same shape as `tutorial/examples/wordle.py`:

1. Reset one fresh in-process env per dataset prompt.
2. Apply the chat template, call `generate_rollout_completions(trainer, [prompt_text])`.
3. **Parse the model completion text into a typed `NegotiationAction`** (this is what makes per-completion variance reach the env).
4. Step the env, accumulate token ids + four verifiable rewards.
5. Return parallel lists for the four reward functions to read from `**kwargs`.

In [ ]:
from typing import Any

from server.environment import SMELiquidityEnvironment
from rl.bridge import format_observation, parse_action
from sme_negotiator_env.graders import compute_verifiable_reward_breakdown

SYSTEM_PROMPT = (
    "You are an SME treasury agent operating a long-horizon liquidity workflow with a hostile buyer. "
    "On every turn you MUST reply with a single JSON object describing one action. Do NOT include any "
    "prose outside the JSON. The schema is:\n"
    "{\n"
    '  "action_type": "propose" | "accept" | "reject" | "tool" | "simulate_plan" | "advance_period",\n'
    '  "deal_id": "<id from open_deal_ids>",\n'
    '  "price": <float, must stay >= cost_threshold>,\n'
    '  "payment_days": <int, lower is better, target liquidity_threshold>,\n'
    '  "use_treds": true | false,\n'
    '  "propose_late_payment_penalty_clause": true | false,\n'
    '  "propose_dynamic_discounting": true | false,\n'
    '  "dynamic_discount_annual_rate": <0.0-0.05>,\n'
    '  "reason": "<short rationale>"\n'
    "}\n"
    "Goals: avoid SME default, drive payment_days down toward liquidity_threshold, and improve NPV vs the baseline."
)


def _build_user_prompt(observation: Any) -> str:
    return (
        f"Episode state:\n{format_observation(observation)}\n\n"
        "Reply with ONE JSON action object now."
    )


def _format_score(text: str) -> float:
    """Bounded text-quality reward in [0, 1] so GRPO advantage is non-zero before the env trajectory responds."""
    raw = (text or "").strip()
    if not raw:
        return 0.0
    score = 0.0
    if raw.startswith("{") and raw.rstrip().endswith("}"):
        score += 0.3
    if '"action_type"' in raw:
        score += 0.15
    for key in ("payment_days", "price", "use_treds"):
        if key in raw:
            score += 0.07
    if 50 <= len(raw) <= 600:
        score += 0.10
    return float(min(1.0, score))


def rollout_once(trainer, tokenizer, dataset_prompt: str) -> dict:
    """Run one full liquidity episode driven by the model and return GRPO-ready signals."""
    from trl.experimental.openenv import generate_rollout_completions

    env = SMELiquidityEnvironment(total_periods=TOTAL_PERIODS)
    observation = env.reset(seed=hash(dataset_prompt) % (2**31), difficulty=DIFFICULTY, task_name=TASK_NAME)

    prompt_ids:    list[int]   = []
    completion_ids: list[int]  = []
    logprobs:      list[float] = []
    last_text = ""
    invalid_steps = 0

    for turn in range(MAX_TURNS):
        if observation.done:
            break
        if not observation.open_deal_ids and observation.current_period < observation.total_periods:
            from sme_negotiator_env.models import NegotiationAction
            observation = env.step(NegotiationAction(action_type="advance_period"))
            continue

        chat = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": _build_user_prompt(observation)},
        ]
        prompt_text = tokenizer.apply_chat_template(chat, add_generation_prompt=True, tokenize=False)
        rollout = generate_rollout_completions(trainer, [prompt_text])[0]
        prompt_ids.extend(rollout["prompt_ids"])
        completion_ids.extend(rollout["completion_ids"])
        logprobs.extend(rollout["logprobs"])
        completion_text = rollout.get("text") or tokenizer.decode(rollout["completion_ids"], skip_special_tokens=True)
        last_text = completion_text

        try:
            action = parse_action(completion_text, observation)
            observation = env.step(action)
        except Exception:
            invalid_steps += 1
            from sme_negotiator_env.models import NegotiationAction
            observation = env.step(NegotiationAction(action_type="advance_period"))

    # Derive the verifiable reward from the env's WorldState across all resolved deals.
    state = env.state
    world_state = state.world_state
    weighted = {"solvency": 0.0, "liquidity": 0.0, "npv": 0.0, "compliance": 0.0, "total": 0.0}
    weight_sum = 0.0
    for deal in world_state.deals:
        if deal.status == "open":
            continue
        traj = state.deal_trajectories.get(deal.deal_id) or []
        if not traj:
            continue
        bd = compute_verifiable_reward_breakdown(world_state, list(traj))
        w = float(deal.invoice_amount) if float(deal.invoice_amount) > 0 else 1.0
        weighted["solvency"]   += w * float(bd.solvency)
        weighted["liquidity"]  += w * float(bd.liquidity)
        weighted["npv"]        += w * float(bd.npv)
        weighted["compliance"] += w * float(bd.compliance)
        weighted["total"]      += w * float(bd.total)
        weight_sum += w
    if weight_sum > 0:
        for k in weighted:
            weighted[k] = round(weighted[k] / weight_sum, 6)
    defaulted = any(s.defaulted for s in world_state.smes)
    if defaulted:
        for k in weighted:
            weighted[k] = 0.0

    return {
        "prompt_ids":      prompt_ids,
        "completion_ids":  completion_ids,
        "logprobs":        logprobs,
        "reward_total":      float(weighted["total"]),
        "reward_solvency":   float(weighted["solvency"]),
        "reward_npv":        float(weighted["npv"]),
        "reward_compliance": float(weighted["compliance"]),
        "reward_format":     _format_score(last_text),
        "invalid_steps":     int(invalid_steps),
    }

## 6. Multiple verifiable reward functions (canonical TRL pattern)

Following the *Reward Engineering* guidance (https://arxiv.org/abs/2408.10215, https://arxiv.org/abs/2410.19100):

* `reward_total`     — the canonical 0.35/0.20/0.35/0.10 RLVR composite (dominant signal).
* `reward_solvency`  — anti-default safety check (penalises destructive actions).
* `reward_npv`       — NPV uplift vs the SME's status-quo baseline.
* `reward_format`    — bounded text-quality term that guarantees per-group variance so GRPO advantage ≠ 0 from step 1.

Reward weights match the kube-sre-gym style: outcome reward dominant, format reward bounded.

In [ ]:
def reward_total(completions, **kwargs):
    rewards = kwargs.get("reward_total")
    return [float(r) for r in rewards] if rewards is not None else [0.0 for _ in completions]

def reward_solvency(completions, **kwargs):
    rewards = kwargs.get("reward_solvency")
    return [float(r) for r in rewards] if rewards is not None else [0.0 for _ in completions]

def reward_npv(completions, **kwargs):
    rewards = kwargs.get("reward_npv")
    return [float(r) for r in rewards] if rewards is not None else [0.0 for _ in completions]

def reward_compliance(completions, **kwargs):
    rewards = kwargs.get("reward_compliance")
    return [float(r) for r in rewards] if rewards is not None else [0.0 for _ in completions]

def reward_format(completions, **kwargs):
    rewards = kwargs.get("reward_format")
    return [float(r) for r in rewards] if rewards is not None else [0.0 for _ in completions]

REWARD_FUNCS   = [reward_total, reward_solvency, reward_npv, reward_compliance, reward_format]
REWARD_WEIGHTS = [1.0,          0.3,             0.3,        0.2,               0.1]

## 7. Build dataset, tokenizer, and `GRPOConfig`

Same shape as the canonical Wordle script (`tutorial/examples/wordle.py`).

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer
from trl import GRPOConfig
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

DATASET_PROMPT = "Negotiate the SME's payment terms across the macro horizon while staying solvent."
dataset = Dataset.from_dict({"prompt": [DATASET_PROMPT] * DATASET_SIZE})

torch_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else (torch.float16 if torch.cuda.is_available() else None)

grpo_config = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    max_steps=TRAIN_STEPS,
    learning_rate=5e-6,
    weight_decay=0.0,
    warmup_steps=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_NEW_TOKENS,
    temperature=0.9,
    top_p=0.95,
    logging_steps=1,
    save_strategy="steps",
    save_steps=max(1, TRAIN_STEPS),
    save_total_limit=1,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=(torch_dtype == torch.bfloat16),
    fp16=(torch_dtype == torch.float16),
    reward_weights=REWARD_WEIGHTS,
)
print("GRPOConfig ready. dtype=", torch_dtype)

## 8. Wire `rollout_func` and start GRPO

The `rollout_func` produces parallel lists for the five reward functions exactly like `tutorial/examples/wordle.py`. We also use **PEFT/LoRA** so this fits a free-tier T4. Inference dominates RL runtime (per OpenEnv guidance), so a tiny 8-prompt micro-batch keeps the loop tight.

In [ ]:
from transformers import AutoModelForCausalLM, TrainerCallback
from trl import GRPOTrainer

model_kwargs = {"low_cpu_mem_usage": True}
if torch_dtype is not None:
    model_kwargs["torch_dtype"] = torch_dtype
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
if hasattr(model, "config"):
    model.config.use_cache = False
if hasattr(model, "gradient_checkpointing_enable"):
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

try:
    from peft import LoraConfig, get_peft_model
    lora = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()
except ImportError:
    print("[warn] peft not installed; full-parameter training fallback.")


def rollout_func(prompts, trainer):
    """Per-prompt rollout returning parallel reward lists."""
    out = {
        "prompt_ids":        [],
        "completion_ids":    [],
        "logprobs":          [],
        "reward_total":      [],
        "reward_solvency":   [],
        "reward_npv":        [],
        "reward_compliance": [],
        "reward_format":     [],
    }
    for prompt_text in prompts:
        ep = rollout_once(trainer, tokenizer, prompt_text if isinstance(prompt_text, str) else str(prompt_text))
        out["prompt_ids"].append(ep["prompt_ids"])
        out["completion_ids"].append(ep["completion_ids"])
        out["logprobs"].append(ep["logprobs"])
        out["reward_total"].append(ep["reward_total"])
        out["reward_solvency"].append(ep["reward_solvency"])
        out["reward_npv"].append(ep["reward_npv"])
        out["reward_compliance"].append(ep["reward_compliance"])
        out["reward_format"].append(ep["reward_format"])
    return out


class RewardCurveCallback(TrainerCallback):
    """Capture per-step (avg) reward for the post-training curve."""
    def __init__(self):
        self.steps = []
        self.avg_total = []
        self.avg_solvency = []
        self.avg_npv = []
        self.avg_compliance = []
        self.avg_format = []
        self.loss = []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return control
        if "loss" in logs:
            self.loss.append(float(logs["loss"]))
        # TRL logs each reward function as e.g. "rewards/reward_total/mean".
        for key, dest in (
            ("rewards/reward_total/mean",      self.avg_total),
            ("rewards/reward_solvency/mean",   self.avg_solvency),
            ("rewards/reward_npv/mean",        self.avg_npv),
            ("rewards/reward_compliance/mean", self.avg_compliance),
            ("rewards/reward_format/mean",     self.avg_format),
        ):
            if key in logs:
                dest.append(float(logs[key]))
        if any(k.startswith("rewards/") for k in logs):
            self.steps.append(int(state.global_step))
        return control

reward_callback = RewardCurveCallback()

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=REWARD_FUNCS,
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
    callbacks=[reward_callback],
)
trainer.train()

checkpoint_path = OUTPUT_DIR / "final-grpo-model"
trainer.save_model(str(checkpoint_path))
tokenizer.save_pretrained(str(checkpoint_path))
print(f"checkpoint saved to {checkpoint_path}")

## 9. Reward curves + baseline overlay

Two plots, both labelled with units, both saved as `.png` so judges can pull them straight from the repo.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax_r, ax_l) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
if reward_callback.steps:
    ax_r.plot(reward_callback.steps, reward_callback.avg_total,      label="reward_total (RLVR composite)")
    ax_r.plot(reward_callback.steps, reward_callback.avg_solvency,   label="reward_solvency")
    ax_r.plot(reward_callback.steps, reward_callback.avg_npv,        label="reward_npv")
    ax_r.plot(reward_callback.steps, reward_callback.avg_compliance, label="reward_compliance")
    ax_r.plot(reward_callback.steps, reward_callback.avg_format,     label="reward_format (bounded)", linestyle=":")
ax_r.axhline(y=baseline.get(TASK_NAME, 0.0), color="red", linestyle="--", linewidth=1.0, label=f"baseline ({TASK_NAME})")
ax_r.set_ylabel("Avg reward (per GRPO step)")
ax_r.set_title("GRPO training curves — SME Negotiator (Theme #1)")
ax_r.legend(loc="best", fontsize=8)
ax_r.grid(alpha=0.3)

if reward_callback.loss:
    ax_l.plot(range(1, len(reward_callback.loss) + 1), reward_callback.loss, label="GRPO loss", color="black")
ax_l.set_xlabel("GRPO step")
ax_l.set_ylabel("Loss")
ax_l.legend(loc="best", fontsize=8)
ax_l.grid(alpha=0.3)
fig.tight_layout()

curve_path = OUTPUT_DIR / "reward_curve.png"
fig.savefig(curve_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"saved {curve_path}")

## 10. Before / after evaluation — trained policy vs heuristic baseline

The model is loaded back via `peft.PeftModel` so the LoRA adapter is actually applied (this matters — see OpenEnv tip #16, never naively merge a LoRA adapter).

In [ ]:
from transformers import AutoModelForCausalLM
import torch

try:
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch_dtype, low_cpu_mem_usage=True)
    trained_model = PeftModel.from_pretrained(base, str(checkpoint_path))
    trained_model.eval()
except Exception as exc:
    print("[warn] PeftModel load failed; falling back to plain HF load:", exc)
    trained_model = AutoModelForCausalLM.from_pretrained(str(checkpoint_path), torch_dtype=torch_dtype)
    trained_model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
trained_model.to(device)


def evaluate_trained_policy(seed: int = 7) -> dict:
    env = SMELiquidityEnvironment(total_periods=TOTAL_PERIODS)
    observation = env.reset(seed=seed, difficulty=DIFFICULTY, task_name=TASK_NAME)
    for _ in range(MAX_TURNS):
        if observation.done:
            break
        if not observation.open_deal_ids and observation.current_period < observation.total_periods:
            from sme_negotiator_env.models import NegotiationAction
            observation = env.step(NegotiationAction(action_type="advance_period"))
            continue
        chat = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": _build_user_prompt(observation)},
        ]
        prompt_text = tokenizer.apply_chat_template(chat, add_generation_prompt=True, tokenize=False)
        inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=1400).to(device)
        with torch.no_grad():
            generated = trained_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        completion = tokenizer.decode(generated[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        try:
            action = parse_action(completion, observation)
            observation = env.step(action)
        except Exception:
            from sme_negotiator_env.models import NegotiationAction
            observation = env.step(NegotiationAction(action_type="advance_period"))

    state = env.state
    world = state.world_state
    weighted_total = 0.0
    weight_sum = 0.0
    for deal in world.deals:
        if deal.status == "open":
            continue
        traj = state.deal_trajectories.get(deal.deal_id) or []
        if not traj:
            continue
        bd = compute_verifiable_reward_breakdown(world, list(traj))
        w = float(deal.invoice_amount) if float(deal.invoice_amount) > 0 else 1.0
        weighted_total += w * float(bd.total)
        weight_sum += w
    return {
        "trained_avg_verifiable_reward": round(weighted_total / max(weight_sum, 1.0), 6),
        "defaulted_sme_count":           sum(1 for s in world.smes if s.defaulted),
        "resolved_deal_count":           len(state.resolved_deal_ids),
    }

trained_summary = evaluate_trained_policy(seed=7)
print("BEFORE (heuristic baseline) :", baseline)
print("AFTER  (trained policy)     :", trained_summary)

## 11. (Optional) Publish checkpoint + model card to the Hugging Face Hub

Set `HF_REPO_ID` to your namespace, then run.

In [ ]:
from huggingface_hub import HfApi
from pathlib import Path

HF_REPO_ID = "YOUR_USERNAME/openenv-sme-negotiator-grpo"
CARD_PATH  = Path("huggingface/model_card.md")

if not HF_REPO_ID.startswith("YOUR_USERNAME/") and CARD_PATH.exists():
    api = HfApi()
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
    api.upload_folder(
        repo_id=HF_REPO_ID,
        repo_type="model",
        folder_path=str(checkpoint_path),
        ignore_patterns=["*.tmp", "*.log"],
    )
    api.upload_file(
        repo_id=HF_REPO_ID,
        repo_type="model",
        path_or_fileobj=str(CARD_PATH),
        path_in_repo="README.md",
    )
    print(f"published https://huggingface.co/{HF_REPO_ID}")
else:
    print("Skipping Hub publish (HF_REPO_ID not set or model_card.md missing).")